In [1]:
"""
Примеры кода к Лекции 5.2: State Management — состояние как фундамент графа

Этот модуль демонстрирует:
1. Определение состояния через TypedDict, reducer'ы, Overwrite
2. add_messages и MessagesState
3. Обновление состояния из узлов (частичный возврат)
4. Валидация состояния через Pydantic
5. Подграфы и вложенные состояния
6. Приватное состояние (input/output schema)
7. Практический пример: агент-исследователь с итерациями
"""

"\nПримеры кода к Лекции 5.2: State Management — состояние как фундамент графа\n\nЭтот модуль демонстрирует:\n1. Определение состояния через TypedDict, reducer'ы, Overwrite\n2. add_messages и MessagesState\n3. Обновление состояния из узлов (частичный возврат)\n4. Валидация состояния через Pydantic\n5. Подграфы и вложенные состояния\n6. Приватное состояние (input/output schema)\n7. Практический пример: агент-исследователь с итерациями\n"

In [2]:
import operator
from typing import Annotated, TypedDict

from langchain_core.messages import (
    AIMessage,
    HumanMessage,
    RemoveMessage,
    SystemMessage,
)
from langgraph.graph import END, START, MessagesState, StateGraph
from langgraph.graph.message import add_messages
from llm_config import check_api_key, get_llm

In [3]:
if not check_api_key():
    raise ValueError("API key is not set")
else:
    print("API key is set")

API key is set


In [4]:
# ============================================================================
# ЧАСТЬ 1: ОПРЕДЕЛЕНИЕ СОСТОЯНИЯ — TypedDict И ANNOTATION
# ============================================================================
def example_typed_dict_state():
    """
    Базовое определение состояния через TypedDict.

    Показывает, что без reducer'ов все поля обновляются
    полной заменой: новое значение затирает старое.
    """
    print("=" * 60)
    print("ПРИМЕР 1: TypedDict-состояние (полная замена полей)")
    print("=" * 60)

    class ResearchState(TypedDict):
        question: str
        sources: list[str]
        answer: str

    # Узел A записывает sources = ["wiki"]
    def node_a(state: ResearchState) -> dict:
        print("  Узел A: записываю sources = ['wiki']")
        return {"sources": ["wiki"]}

    # Узел B записывает sources = ["arxiv"] — без reducer'а затрёт результат A
    def node_b(state: ResearchState) -> dict:
        print("  Узел B: записываю sources = ['arxiv']")
        print(f"  Узел B: текущее значение sources = {state['sources']}")
        return {"sources": ["arxiv"]}

    graph = StateGraph(ResearchState)
    graph.add_node("node_a", node_a)
    graph.add_node("node_b", node_b)
    graph.add_edge(START, "node_a")
    graph.add_edge("node_a", "node_b")
    graph.add_edge("node_b", END)
    app = graph.compile()

    result = app.invoke({
        "question": "Что такое квантовая запутанность?",
        "sources": [],
        "answer": "",
    })
    print(f"Финальное sources: {result['sources']}")
    print("Результат 'wiki' потерян — это поведение по умолчанию без reducer'а.\n")


if __name__ == "__main__":
    example_typed_dict_state()

ПРИМЕР 1: TypedDict-состояние (полная замена полей)
  Узел A: записываю sources = ['wiki']
  Узел B: записываю sources = ['arxiv']
  Узел B: текущее значение sources = ['wiki']
Финальное sources: ['arxiv']
Результат 'wiki' потерян — это поведение по умолчанию без reducer'а.



In [5]:
def example_reducers():
    """
    Состояние с reducer'ами через Annotated.

    Демонстрирует разные типы reducer'ов в реальном графе:
    - operator.add для конкатенации списков
    - lambda для кастомной логики (счётчик)
    Сравните с Примером 1: там без reducer'а значение затиралось,
    здесь — накапливается.
    """
    print("=" * 60)
    print("ПРИМЕР 2: Reducer'ы через Annotated")
    print("=" * 60)

    class ResearchState(TypedDict):
        question: str  # Замена (по умолчанию)
        sources: Annotated[list[str], operator.add]  # Дополнение списка
        attempt_count: Annotated[
            int, lambda old, new: old + new
        ]  # Кастомная логика: суммирование

    # Два узла пишут в одни и те же поля — reducer'ы объединяют результаты
    def node_a(state: ResearchState) -> dict:
        print("  Узел A: sources=['wiki'], attempt_count=1")
        return {"sources": ["wiki"], "attempt_count": 1}

    def node_b(state: ResearchState) -> dict:
        print("  Узел B: sources=['arxiv'], attempt_count=1")
        return {"sources": ["arxiv"], "attempt_count": 1}

    graph = StateGraph(ResearchState)
    graph.add_node("node_a", node_a)
    graph.add_node("node_b", node_b)
    graph.add_edge(START, "node_a")
    graph.add_edge("node_a", "node_b")
    graph.add_edge("node_b", END)
    app = graph.compile()

    result = app.invoke({
        "question": "Квантовая запутанность",
        "sources": [],
        "attempt_count": 0,
    })
    print(f"\nФинальные sources: {result['sources']}")
    print("  → operator.add: ['wiki'] + ['arxiv'] = ['wiki', 'arxiv']")
    print(f"Финальный attempt_count: {result['attempt_count']}")
    print("  → lambda old, new: old + new: 0 + 1 + 1 = 2\n")


if __name__ == "__main__":
    example_reducers()

ПРИМЕР 2: Reducer'ы через Annotated
  Узел A: sources=['wiki'], attempt_count=1
  Узел B: sources=['arxiv'], attempt_count=1

Финальные sources: ['wiki', 'arxiv']
  → operator.add: ['wiki'] + ['arxiv'] = ['wiki', 'arxiv']
Финальный attempt_count: 2
  → lambda old, new: old + new: 0 + 1 + 1 = 2



In [6]:
def example_overwrite():
    """
    Overwrite из langgraph.types позволяет обойти reducer и записать
    значение напрямую, полностью заменив текущее.

    Это полезно, когда у поля есть reducer (например, operator.add для
    накопления списка), но в определённом узле нужно сбросить накопленное
    и начать заново.

    Демонстрирует:
    - Поле с reducer'ом operator.add (нормальное накопление)
    - Один узел добавляет элементы через reducer (дополнение)
    - Другой узел использует Overwrite для полной замены списка
    """
    from langgraph.types import Overwrite

    print("=" * 60)
    print("ПРИМЕР: Overwrite — обход reducer'а для прямой записи")
    print("=" * 60)

    # Состояние с reducer'ом operator.add для накопления элементов
    class CollectorState(TypedDict):
        items: Annotated[list[str], operator.add]
        stage: str

    def collect_first(state: CollectorState) -> dict:
        """Первый узел: добавляет элементы через reducer (дополнение)."""
        return {
            "items": ["alpha", "beta"],
            "stage": "collected",
        }

    def collect_second(state: CollectorState) -> dict:
        """Второй узел: тоже добавляет через reducer — список растёт."""
        return {
            "items": ["gamma"],
            "stage": "extended",
        }

    def reset_with_overwrite(state: CollectorState) -> dict:
        """Третий узел: использует Overwrite — полностью заменяет список,
        игнорируя reducer operator.add."""
        # Без Overwrite: {"items": ["only_this"]} добавит "only_this" к существующему списку
        # С Overwrite: список будет заменён на ["only_this"]
        return {
            "items": Overwrite(["only_this"]),
            "stage": "reset",
        }

    # --- Граф 1: обычное накопление (без Overwrite) ---
    graph_normal = StateGraph(CollectorState)
    graph_normal.add_node("first", collect_first)
    graph_normal.add_node("second", collect_second)
    graph_normal.add_edge(START, "first")
    graph_normal.add_edge("first", "second")
    graph_normal.add_edge("second", END)

    app_normal = graph_normal.compile()
    result_normal = app_normal.invoke({"items": [], "stage": "init"})

    print(f"\nОбычное накопление (reducer operator.add):")
    print(f"  Результат items: {result_normal['items']}")
    print(f"  → ['alpha', 'beta'] + ['gamma'] = ['alpha', 'beta', 'gamma']")

    # --- Граф 2: с использованием Overwrite ---
    graph_overwrite = StateGraph(CollectorState)
    graph_overwrite.add_node("first", collect_first)
    graph_overwrite.add_node("second", collect_second)
    graph_overwrite.add_node("reset", reset_with_overwrite)
    graph_overwrite.add_edge(START, "first")
    graph_overwrite.add_edge("first", "second")
    graph_overwrite.add_edge("second", "reset")
    graph_overwrite.add_edge("reset", END)

    app_overwrite = graph_overwrite.compile()
    result_overwrite = app_overwrite.invoke({"items": [], "stage": "init"})

    print(f"\nС использованием Overwrite:")
    print(f"  Результат items: {result_overwrite['items']}")
    print(f"  → Overwrite(['only_this']) заменил ['alpha', 'beta', 'gamma'] целиком")
    print(f"  Reducer operator.add был обойдён — список полностью перезаписан\n")


if __name__ == "__main__":
    example_overwrite()

ПРИМЕР: Overwrite — обход reducer'а для прямой записи

Обычное накопление (reducer operator.add):
  Результат items: ['alpha', 'beta', 'gamma']
  → ['alpha', 'beta'] + ['gamma'] = ['alpha', 'beta', 'gamma']

С использованием Overwrite:
  Результат items: ['only_this']
  → Overwrite(['only_this']) заменил ['alpha', 'beta', 'gamma'] целиком
  Reducer operator.add был обойдён — список полностью перезаписан



In [7]:
# ============================================================================
# ЧАСТЬ 2: add_messages — ГЛАВНЫЙ REDUCER В АГЕНТНЫХ ПРИЛОЖЕНИЯХ
# ============================================================================
def example_add_messages():
    """
    Демонстрация add_messages: добавление, обновление по id, удаление.

    add_messages — специальный reducer из LangGraph, который умнее
    простого operator.add: обрабатывает дубликаты по id и поддерживает
    удаление через RemoveMessage.
    """
    print("=" * 60)
    print("ПРИМЕР 3: add_messages — три режима работы")
    print("=" * 60)

    # --- Режим 1: базовое добавление (разные id) ---
    old = [HumanMessage(content="Привет", id="1")]
    new = [AIMessage(content="Здравствуйте!", id="2")]
    result = add_messages(old, new)
    print("Добавление:")
    for msg in result:
        print(f"  {msg.__class__.__name__}(id={msg.id}): {msg.content}")

    # --- Режим 2: обновление по id (одинаковый id) ---
    old = [AIMessage(content="Черновик ответа", id="2")]
    new = [AIMessage(content="Финальный ответ", id="2")]
    result = add_messages(old, new)
    print("\nОбновление по id:")
    for msg in result:
        print(f"  {msg.__class__.__name__}(id={msg.id}): {msg.content}")
    print("  → Заменено, не продублировано!")

    # --- Режим 3: удаление через RemoveMessage ---
    old = [
        HumanMessage(content="Привет", id="1"),
        AIMessage(content="Ответ", id="2"),
    ]
    new = [RemoveMessage(id="1")]
    result = add_messages(old, new)
    print("\nУдаление:")
    for msg in result:
        print(f"  {msg.__class__.__name__}(id={msg.id}): {msg.content}")
    print("  → Первое сообщение удалено\n")


if __name__ == "__main__":
    example_add_messages()

ПРИМЕР 3: add_messages — три режима работы
Добавление:
  HumanMessage(id=1): Привет
  AIMessage(id=2): Здравствуйте!

Обновление по id:
  AIMessage(id=2): Финальный ответ
  → Заменено, не продублировано!

Удаление:
  AIMessage(id=2): Ответ
  → Первое сообщение удалено



In [8]:
def example_messages_state():
    """
    MessagesState — готовый шаблон для состояния с сообщениями.

    Если нужны дополнительные поля помимо messages — наследуемся
    от MessagesState и добавляем свои поля.
    """
    print("=" * 60)
    print("ПРИМЕР 4: MessagesState и наследование")
    print("=" * 60)

    # MessagesState эквивалентен:
    # class MessagesState(TypedDict):
    #     messages: Annotated[list[BaseMessage], add_messages]

    # Наследуемся и добавляем свои поля
    class AgentState(MessagesState):
        current_plan: str
        tools_used: Annotated[list[str], operator.add]
        iteration: int

    print("AgentState наследует messages от MessagesState")
    print(f"Аннотации AgentState: {AgentState.__annotations__}")
    print()


if __name__ == "__main__":
    example_messages_state()

ПРИМЕР 4: MessagesState и наследование
AgentState наследует messages от MessagesState
Аннотации AgentState: {'messages': ForwardRef('Annotated[list[AnyMessage], add_messages]', module='langgraph.graph.message'), 'current_plan': <class 'str'>, 'tools_used': typing.Annotated[list[str], <built-in function add>], 'iteration': <class 'int'>}



In [9]:
# ============================================================================
# ЧАСТЬ 3: КАК УЗЛЫ ОБНОВЛЯЮТ СОСТОЯНИЕ
# ============================================================================
def example_partial_updates():
    """
    Узел возвращает только изменения, а не полное состояние.

    Каждый узел — функция, принимающая полное состояние и возвращающая
    словарь только с изменёнными полями. Не упомянутые поля остаются
    без изменений.
    """
    print("=" * 60)
    print("ПРИМЕР 5: Частичные обновления состояния")
    print("=" * 60)

    class PlannerState(TypedDict):
        goal: str
        plan: list[str]
        current_step: int
        result: str

    def create_plan(state: PlannerState) -> dict:
        """Узел обновляет только plan и current_step. goal и result не трогает."""
        print(f"  create_plan: goal='{state['goal']}' (не трогаем)")
        return {
            "plan": ["Шаг 1: поиск", "Шаг 2: анализ", "Шаг 3: ответ"],
            "current_step": 0,
        }

    def execute_step(state: PlannerState) -> dict:
        """Узел обновляет current_step и result."""
        step = state["plan"][state["current_step"]]
        print(f"  execute_step: выполняю '{step}', goal='{state['goal']}' (не тронут)")
        return {
            "current_step": state["current_step"] + 1,
            "result": f"Выполнен: {step}",
        }

    graph = StateGraph(PlannerState)
    graph.add_node("create_plan", create_plan)
    graph.add_node("execute_step", execute_step)
    graph.add_edge(START, "create_plan")
    graph.add_edge("create_plan", "execute_step")
    graph.add_edge("execute_step", END)
    app = graph.compile()

    result = app.invoke({
        "goal": "Ответить на вопрос",
        "plan": [],
        "current_step": 0,
        "result": "",
    })
    print("\nФинальное состояние:")
    print(f"  goal='{result['goal']}' (не тронут ни одним узлом)")
    print(f"  plan={result['plan']}")
    print(f"  current_step={result['current_step']}")
    print(f"  result='{result['result']}'")
    print()


if __name__ == "__main__":
    example_partial_updates()

ПРИМЕР 5: Частичные обновления состояния
  create_plan: goal='Ответить на вопрос' (не трогаем)
  execute_step: выполняю 'Шаг 1: поиск', goal='Ответить на вопрос' (не тронут)

Финальное состояние:
  goal='Ответить на вопрос' (не тронут ни одним узлом)
  plan=['Шаг 1: поиск', 'Шаг 2: анализ', 'Шаг 3: ответ']
  current_step=1
  result='Выполнен: Шаг 1: поиск'



In [10]:
# ============================================================================
# ЧАСТЬ 4: ВАЛИДАЦИЯ СОСТОЯНИЯ ЧЕРЕЗ PYDANTIC
# ============================================================================
def example_pydantic_validation():
    """
    Pydantic-состояние с валидацией типов и бизнес-правил.

    В отличие от TypedDict, Pydantic проверяет типы в runtime,
    поддерживает значения по умолчанию и кастомные валидаторы.
    """
    from pydantic import BaseModel, Field, field_validator

    print("=" * 60)
    print("ПРИМЕР 6: Pydantic-валидация состояния")
    print("=" * 60)

    class ReviewState(BaseModel):
        text: str = Field(description="Текст для ревью")
        issues: list[str] = Field(
            default_factory=list, description="Найденные проблемы"
        )
        score: float = Field(
            default=0.0, ge=0.0, le=10.0, description="Оценка от 0 до 10"
        )
        approved: bool = Field(default=False)

        @field_validator("score")
        @classmethod
        def score_must_be_reasonable(cls, v):
            if v < 0 or v > 10:
                raise ValueError("Оценка должна быть от 0 до 10")
            return round(v, 1)

    # Создание состояния со значениями по умолчанию
    state = ReviewState(text="Пример текста")
    print(f"Состояние со значениями по умолчанию:")
    print(f"  text='{state.text}', score={state.score}, issues={state.issues}")

    # Валидация: score округляется до 1 знака
    state2 = ReviewState(text="Другой текст", score=7.456)
    print(f"  score=7.456 → {state2.score} (округлено валидатором)")

    # Валидация: score вне диапазона — ошибка
    try:
        ReviewState(text="Тест", score=15.0)
    except Exception as e:
        print(f"  score=15.0 → Ошибка: {e}")

    # Использование с StateGraph
    graph = StateGraph(ReviewState)
    print("\nStateGraph(ReviewState) — граф с Pydantic-валидацией создан успешно")

    # ВАЖНО: Выход графа НЕ является экземпляром Pydantic-модели
    # Runtime-валидация происходит только на входе первого узла, но не между узлами
    print("\n--- Важные ограничения Pydantic-состояния ---")
    print(
        "ВАЖНО: Выход графа НЕ является экземпляром Pydantic-модели — это обычный dict"
    )
    print(
        "Runtime-валидация происходит только на входе первого узла, но не между узлами"
    )
    print("Рекомендация: TypedDict обычно предпочтительнее для частичных обновлений\n")


if __name__ == "__main__":
    example_pydantic_validation()

ПРИМЕР 6: Pydantic-валидация состояния
Состояние со значениями по умолчанию:
  text='Пример текста', score=0.0, issues=[]
  score=7.456 → 7.5 (округлено валидатором)
  score=15.0 → Ошибка: 1 validation error for ReviewState
score
  Input should be less than or equal to 10 [type=less_than_equal, input_value=15.0, input_type=float]
    For further information visit https://errors.pydantic.dev/2.12/v/less_than_equal

StateGraph(ReviewState) — граф с Pydantic-валидацией создан успешно

--- Важные ограничения Pydantic-состояния ---
ВАЖНО: Выход графа НЕ является экземпляром Pydantic-модели — это обычный dict
Runtime-валидация происходит только на входе первого узла, но не между узлами
Рекомендация: TypedDict обычно предпочтительнее для частичных обновлений



In [11]:
# ============================================================================
# ЧАСТЬ 5: ВЛОЖЕННЫЕ СОСТОЯНИЯ И ПОДГРАФЫ
# ============================================================================
def example_subgraph():
    """
    Подграф как модуль: собственное состояние, встраивание в родительский граф.

    Подграф (subgraph) имеет своё состояние и логику. Может быть
    вставлен как узел в родительский граф.
    """
    print("=" * 60)
    print("ПРИМЕР 7: Подграф с параллельным выполнением")
    print("=" * 60)

    # === Подграф: исследование ===
    class ResearchState(TypedDict):
        query: str
        findings: Annotated[list[str], operator.add]

    def search_web(state: ResearchState) -> dict:
        """Поиск в интернете."""
        return {"findings": [f"Веб-результат по '{state['query']}'"]}

    def search_docs(state: ResearchState) -> dict:
        """Поиск в документации."""
        return {"findings": [f"Документ по '{state['query']}'"]}

    def summarize_findings(state: ResearchState) -> dict:
        """Суммаризация найденного."""
        summary = "; ".join(state["findings"])
        return {"findings": [f"ИТОГО: {summary}"]}

    # Собираем подграф
    research_graph = StateGraph(ResearchState)
    research_graph.add_node("web", search_web)
    research_graph.add_node("docs", search_docs)
    research_graph.add_node("summarize", summarize_findings)

    research_graph.add_edge(START, "web")
    research_graph.add_edge(START, "docs")
    research_graph.add_edge("web", "summarize")
    research_graph.add_edge("docs", "summarize")
    research_graph.add_edge("summarize", END)

    compiled_research = research_graph.compile()

    # Запускаем подграф отдельно
    result = compiled_research.invoke({"query": "квантовая запутанность"})
    print(f"Результат подграфа: {result['findings']}")
    print()

    return compiled_research


if __name__ == "__main__":
    example_subgraph()

ПРИМЕР 7: Подграф с параллельным выполнением
Результат подграфа: ["Документ по 'квантовая запутанность'", "Веб-результат по 'квантовая запутанность'", "ИТОГО: Документ по 'квантовая запутанность'; Веб-результат по 'квантовая запутанность'"]



In [12]:
def example_subgraph_in_parent():
    """
    Встраивание подграфа в родительский граф двумя способами:
    1. Как узел (автоматический маппинг полей)
    2. Через вызов внутри узла (ручной маппинг)
    """
    print("=" * 60)
    print("ПРИМЕР 8: Подграф внутри родительского графа")
    print("=" * 60)

    # --- Собираем подграф заново ---
    class ResearchState(TypedDict):
        query: str
        findings: Annotated[list[str], operator.add]

    def search_web(state: ResearchState) -> dict:
        return {"findings": [f"Веб-результат по '{state['query']}'"]}

    def search_docs(state: ResearchState) -> dict:
        return {"findings": [f"Документ по '{state['query']}'"]}

    def summarize_findings(state: ResearchState) -> dict:
        summary = "; ".join(state["findings"])
        return {"findings": [f"ИТОГО: {summary}"]}

    research_graph = StateGraph(ResearchState)
    research_graph.add_node("web", search_web)
    research_graph.add_node("docs", search_docs)
    research_graph.add_node("summarize", summarize_findings)
    research_graph.add_edge(START, "web")
    research_graph.add_edge(START, "docs")
    research_graph.add_edge("web", "summarize")
    research_graph.add_edge("docs", "summarize")
    research_graph.add_edge("summarize", END)
    compiled_research = research_graph.compile()

    # --- Способ 1: подграф как узел (автоматический маппинг полей) ---
    # Если у родительского и дочернего состояний есть общие поля,
    # LangGraph маппит их автоматически.
    print("--- Способ 1: add_node(name, compiled_subgraph) ---")

    class ParentState1(TypedDict):
        query: str  # Совпадает с ResearchState.query → маппится автоматически
        findings: Annotated[list[str], operator.add]  # Совпадает → маппится
        final_answer: str

    def answer_from_findings(state: ParentState1) -> dict:
        summary = state["findings"][-1] if state["findings"] else "Ничего не найдено"
        return {"final_answer": f"Ответ: {summary}"}

    parent_graph_1 = StateGraph(ParentState1)
    # Подграф добавляется напрямую — без обёртки в функцию
    parent_graph_1.add_node("research", compiled_research)
    parent_graph_1.add_node("answer", answer_from_findings)
    parent_graph_1.add_edge(START, "research")
    parent_graph_1.add_edge("research", "answer")
    parent_graph_1.add_edge("answer", END)

    app1 = parent_graph_1.compile()
    result1 = app1.invoke({"query": "квантовая запутанность"})
    print(f"Ответ: {result1['final_answer']}")
    print()

    # --- Способ 2: вызов подграфа внутри узла (ручной маппинг) ---
    print("--- Способ 2: invoke подграфа внутри узла ---")

    class ParentState2(TypedDict):
        question: str  # НЕ совпадает с ResearchState.query
        research_results: str
        answer: str

    def research_node(state: ParentState2) -> dict:
        """Вызываем подграф вручную, маппим поля."""
        result = compiled_research.invoke({"query": state["question"]})
        return {"research_results": result["findings"][-1]}  # Берём summary

    def format_answer(state: ParentState2) -> dict:
        """Формирует финальный ответ на основе исследования."""
        return {"answer": f"На основе исследования: {state['research_results']}"}

    main_graph = StateGraph(ParentState2)
    main_graph.add_node("research", research_node)
    main_graph.add_node("answer", format_answer)
    main_graph.add_edge(START, "research")
    main_graph.add_edge("research", "answer")
    main_graph.add_edge("answer", END)

    app2 = main_graph.compile()
    result2 = app2.invoke({"question": "Что такое квантовая запутанность?"})
    print(f"Ответ: {result2['answer']}")
    print()


if __name__ == "__main__":
    example_subgraph_in_parent()

ПРИМЕР 8: Подграф внутри родительского графа
--- Способ 1: add_node(name, compiled_subgraph) ---
Ответ: Ответ: ИТОГО: Документ по 'квантовая запутанность'; Веб-результат по 'квантовая запутанность'

--- Способ 2: invoke подграфа внутри узла ---
Ответ: На основе исследования: ИТОГО: Документ по 'Что такое квантовая запутанность?'; Веб-результат по 'Что такое квантовая запутанность?'



In [13]:
# ============================================================================
# ЧАСТЬ 6: ПРИВАТНОЕ СОСТОЯНИЕ УЗЛА (INPUT/OUTPUT SCHEMA)
# ============================================================================
def example_private_state():
    """
    Input/Output schema: разделение внешнего API и внутреннего состояния.

    Граф принимает InputState, внутри работает с OverallState,
    а возвращает только OutputState. Внутренние поля не видны снаружи.
    """
    print("=" * 60)
    print("ПРИМЕР 9: Приватное состояние (input/output schema)")
    print("=" * 60)

    # Полное состояние — видно внутри графа
    class OverallState(TypedDict):
        question: str  # Входное
        answer: str  # Выходное
        internal_notes: Annotated[list[str], operator.add]  # Внутреннее
        search_cache: dict  # Внутреннее

    # Входная схема — что принимает граф
    class InputState(TypedDict):
        question: str

    # Выходная схема — что возвращает граф
    class OutputState(TypedDict):
        answer: str

    # --- Узел 1: поиск (заполняет внутренние поля) ---
    def search(state: OverallState) -> dict:
        """Ищет информацию, сохраняет в кэш и заметки."""
        query = state["question"]
        cached = {query: f"Результат поиска по '{query}'"}
        return {
            "search_cache": cached,
            "internal_notes": [f"Выполнен поиск: {query}"],
        }

    # --- Узел 2: анализ (читает внутренние поля, пишет новые) ---
    def analyze(state: OverallState) -> dict:
        """Анализирует кэш поиска — использует внутреннее состояние."""
        cached_result = list(state["search_cache"].values())[0]
        return {
            "internal_notes": [f"Проанализировано: {cached_result}"],
        }

    # --- Узел 3: ответ (формирует выходное поле) ---
    def respond(state: OverallState) -> dict:
        """Формирует ответ на основе внутренних заметок."""
        steps = " → ".join(state["internal_notes"])
        return {"answer": f"Ответ (путь: {steps})"}

    graph = StateGraph(
        OverallState,
        input_schema=InputState,
        output_schema=OutputState,
    )
    graph.add_node("search", search)
    graph.add_node("analyze", analyze)
    graph.add_node("respond", respond)
    graph.add_edge(START, "search")
    graph.add_edge("search", "analyze")
    graph.add_edge("analyze", "respond")
    graph.add_edge("respond", END)

    app = graph.compile()

    # Вызов: передаём только question, получаем только answer
    result = app.invoke({"question": "Как работает LangGraph?"})
    print("Вход:  {'question': 'Как работает LangGraph?'}")
    print(f"Выход: {result}")
    print()
    print("search_cache и internal_notes использовались внутри,")
    print("но наружу вернулось только поле answer (OutputState).\n")


if __name__ == "__main__":
    example_private_state()

ПРИМЕР 9: Приватное состояние (input/output schema)
Вход:  {'question': 'Как работает LangGraph?'}
Выход: {'answer': "Ответ (путь: Выполнен поиск: Как работает LangGraph? → Проанализировано: Результат поиска по 'Как работает LangGraph?')"}

search_cache и internal_notes использовались внутри,
но наружу вернулось только поле answer (OutputState).



In [14]:
# ============================================================================
# ЧАСТЬ 7: ПРАКТИЧЕСКИЙ ПРИМЕР — АГЕНТ-ИССЛЕДОВАТЕЛЬ
# ============================================================================
def example_research_agent():
    """
    Полный агент-исследователь с итеративным циклом.

    Агент получает вопрос, планирует подход, ищет информацию,
    оценивает качество и итерирует при необходимости (до 3 раз).

    Демонстрирует:
    - reducer operator.add для накопления findings
    - условную маршрутизацию через add_conditional_edges
    - цикл research → evaluate → research
    - ограничение числа итераций

    ПРИМЕЧАНИЕ: для запуска требуется API-ключ (см. llm_config).
    """
    print("=" * 60)
    print("ПРИМЕР 10: Агент-исследователь с итерациями")
    print("=" * 60)

    llm = get_llm()

    # Входная схема — что принимает граф
    class InputState(TypedDict):
        question: str

    # Выходная схема — что возвращает граф
    class OutputState(TypedDict):
        final_answer: str

    # Полное состояние — внутренняя кухня графа
    class ResearcherState(InputState, OutputState):
        plan: str
        findings: Annotated[list[str], operator.add]
        evaluation: str
        iteration: int

    def planner(state: ResearcherState) -> dict:
        """Составляет план исследования."""
        print(f"\n[plan] Вопрос: {state['question']}")
        response = llm.invoke(
            [
                SystemMessage(
                    content="Составь краткий план исследования для ответа на вопрос. 2-3 шага."
                ),
                HumanMessage(content=state["question"]),
            ]
        )
        print(f"[plan] План:\n{response.content}\n")
        return {"plan": response.content}

    def researcher(state: ResearcherState) -> dict:
        """Ищет информацию через DuckDuckGo. LLM формулирует запрос."""
        from langchain_community.tools import DuckDuckGoSearchResults

        # search = DuckDuckGoSearchRun() - возвращает единую строку с результатами
        # search = DuckDuckGoSearchResults() - возвращает структурированный список с заголовками, ссылками и сниппетами
        search = DuckDuckGoSearchResults()

        # LLM формулирует поисковый запрос на основе плана и обратной связи
        context = f"План: {state['plan']}"
        if state.get("findings"):
            context += f"\nУже найдено: {state['findings']}"
        if state.get("evaluation"):
            context += f"\nОбратная связь: {state['evaluation']}"

        query_response = llm.invoke(
            [
                SystemMessage(
                    content=(
                        "Сформулируй один короткий поисковый запрос для Google/DuckDuckGo "
                        "чтобы найти информацию по плану. Верни ТОЛЬКО запрос, без пояснений."
                    )
                ),
                HumanMessage(content=f"Вопрос: {state['question']}\n{context}"),
            ]
        )
        search_query = query_response.content.strip()
        print(f"[research] Поисковый запрос: {search_query}")

        # Реальный поиск через DuckDuckGo
        search_results = search.invoke(search_query)
        print(f"Найдено: {search_results[:200]}...\n")

        # в findings приходят структурированные результаты со snippet, title и link.
        return {
            "findings": [search_results],
            "iteration": state.get("iteration", 0) + 1,
        }

    def evaluator(state: ResearcherState) -> dict:
        """Оценивает, достаточно ли найденной информации."""
        response = llm.invoke(
            [
                SystemMessage(
                    content=(
                        "Оцени найденную информацию. Достаточно ли её для полного ответа? "
                        "Ответь: SUFFICIENT или INSUFFICIENT и объясни почему."
                    )
                ),
                HumanMessage(
                    content=f"Вопрос: {state['question']}\nНайдено: {state['findings']}"
                ),
            ]
        )
        print(f"[evaluate] {response.content}\n")
        return {"evaluation": response.content}

    def writer(state: ResearcherState) -> dict:
        """Синтезирует финальный ответ из собранных данных."""
        response = llm.invoke(
            [
                SystemMessage(
                    content="Напиши исчерпывающий ответ на основе собранных данных."
                ),
                HumanMessage(
                    content=f"Вопрос: {state['question']}\nДанные: {state['findings']}"
                ),
            ]
        )
        print(f"[write] Финальный ответ:\n{response.content}\n")
        return {"final_answer": response.content}

    def writer_partial(state: ResearcherState) -> dict:
        """Синтезирует ответ с предупреждением о неполноте данных."""
        response = llm.invoke(
            [
                SystemMessage(
                    content="Напиши ответ на основе собранных данных. "
                    "Предупреди, что данных может быть недостаточно для полного ответа."
                ),
                HumanMessage(
                    content=f"Вопрос: {state['question']}\nДанные: {state['findings']}"
                ),
            ]
        )
        print(f"[write_partial] Ответ (данные неполные):\n{response.content}\n")
        return {"final_answer": response.content}

    def should_continue(state: ResearcherState) -> str:
        """Маршрутизатор: продолжать исследование или писать ответ?"""
        if "INSUFFICIENT" not in state.get(
            "evaluation", ""
        ) and "SUFFICIENT" in state.get("evaluation", ""):
            return "no"  # данных достаточно
        if state.get("iteration", 0) >= 3:
            return "forced"  # лимит итераций исчерпан
        return "yes"  # ищем дальше

    # Собираем граф с input/output schema
    graph = StateGraph(
        ResearcherState,
        input_schema=InputState,
        output_schema=OutputState,
    )

    graph.add_node("plan", planner)
    graph.add_node("research", researcher)
    graph.add_node("evaluate", evaluator)
    graph.add_node("write", writer)
    graph.add_node("write_partial", writer_partial)

    graph.add_edge(START, "plan")
    graph.add_edge("plan", "research")
    graph.add_edge("research", "evaluate")
    graph.add_conditional_edges(
        "evaluate",
        should_continue,
        {
            "no": "write",              # данных достаточно → полный ответ
            "yes": "research",          # ищем дальше
            "forced": "write_partial",  # лимит итераций → ответ с дисклеймером
        },
    )
    graph.add_edge("write", END)
    graph.add_edge("write_partial", END)

    app = graph.compile()
    app.invoke({"question": "Как устроена квантовая запутанность?"})


if __name__ == "__main__":
    example_research_agent()

ПРИМЕР 10: Агент-исследователь с итерациями



[plan] Вопрос: Как устроена квантовая запутанность?


[plan] План:
Краткий план исследования вопроса:

1. **Определить базовые понятия**  
   Разобраться, что такое квантовое состояние, суперпозиция и чем запутанность отличается от обычной корреляции.

2. **Изучить механизм возникновения запутанности**  
   Посмотреть, как запутанные состояния появляются при взаимодействии частиц и почему их нельзя описать независимо друг от друга.

3. **Проверить физические следствия и эксперименты**  
   Рассмотреть неравенства Белла, измерения запутанных частиц и практические применения в квантовой связи и вычислениях.



[research] Поисковый запрос: квантовая запутанность что это суперпозиция неравенства Белла эксперименты применение


Найдено: snippet: January 12, 2026 -Теорема Белла (как её теперь называют) показывает, что вне зависимости от реального наличия в квантово-механической теории неких скрытых параметров, влияющих на любую физиче...



...

[evaluate] INSUFFICIENT

Почему: найденная информация в основном даёт **определение** квантовой запутанности и немного контекста про **неравенства Белла** и эксперименты, но этого недостаточно, чтобы **полностью ответить** на вопрос «Как устроена квантовая запутанность?».

Не хватает:
- объяснения **механизма**: как именно возникает запутанное состояние;
- описания **математической формы** (суперпозиция, неразложимость состояния на состояния отдельных частиц);
- пояснения, **что именно измеряется** и почему результаты коррелируют;
- уточнения, что запутанность **не означает передачу сигнала быстрее света**;
- простого, связного объяснения на уровне «как это работает» для читателя.

Итог: данные полезны, но для полного ответа их недостаточно.



[research] Поисковый запрос: квантовая запутанность как возникает суперпозиция неразложимое состояние объяснение механизм измерение неравенства Белла


Найдено: snippet: January 12, 2026 -Теорема Белла (как её теперь называют) показывает, что вне зависимости от реального наличия в квантово-механической теории неких скрытых параметров, влияющих на любую физиче...



[evaluate] INSUFFICIENT

Почему: найденные фрагменты дают только общие определения и упоминания о неравенствах Белла, скрытых параметрах и некоторых экспериментах. Это помогает понять, **что** квантовая запутанность существует и как её проверяют, но не объясняет **как она устроена** на уровне механизма:  
- что именно значит общее квантовое состояние системы;  
- почему состояние частиц нельзя описать отдельно;  
- как возникает запутанность при взаимодействии;  
- что происходит при измерении и почему корреляции не означают передачу сигнала быстрее света.

Для полного ответа нужно более структурное объяснение с примерами и, желательно, без опоры только на краткие сниппеты.



[write_partial] Ответ (данные неполные):
Квантовая запутанность — это явление, при котором состояния двух или большего числа квантовых объектов оказываются взаимосвязанными так, что их нельзя полностью описать по отдельности. Иными словами, нужно рассматривать систему целиком, а не каждую частицу отдельно.

Что это означает на практике:
- До измерения состояние частицы может быть в суперпозиции.
- У запутанной пары результаты измерений оказываются статистически скоррелированными.
- Эти корреляции могут сохраняться даже если частицы находятся далеко друг от друга.

Как это обычно объясняют:
- Частицы рождаются или взаимодействуют так, что их квантовые состояния становятся общим состоянием.
- При измерении одной частицы мы получаем результат, который связан с результатом измерения другой.
- При этом это не означает простую передачу сигнала быстрее света: в квантовой механике речь идет именно о корреляциях, а не о «связи» в обычном смысле.

Почему это важно:
- Теорема Белла и неравенства 